# 01 — CNN Trained From Scratch

**Project:** Bangladeshi Freshwater Fish Image Classification Using CNN and Transfer Learning
**Dataset:** BFF-15 (15 classes)
**Owner:** Vivek Singh

## What this notebook does

1. Loads the frozen train/val/test splits from `data/splits/`.
2. Uses the shared pipeline in `data_utils.py` (no re-splitting, no re-augmentation).
3. Builds a CNN **from scratch**.
4. Trains with `ModelCheckpoint`, `CSVLogger`, `EarlyStopping`, `ReduceLROnPlateau`.
5. Saves all artifacts into `custom_cnn/`:
   - `custom_cnn/logs/training_log.csv`
   - `custom_cnn/best_model/best.keras`
   - `custom_cnn/plots/`
   - `custom_cnn/metrics/final_metrics.json` and `confusion_matrix.csv`

Notebook 03 will read these to compare against the transfer-learning model.

## Contract (must not change)

- Class mapping: `CLASS_NAMES` from `data_utils`.
- Splits: loaded from `data_utils.load_splits()`.
- Augmentation: `data_utils.get_train_ds()` only.
- Input normalization: `[0, 1]` (already applied inside `data_utils`).
- Image size: 224 × 224.
- Seed: 42.

The two model notebooks must use identical data so the comparison is valid.

In [1]:
import os, sys, json, random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from data_utils import (
    get_train_ds, get_val_ds, get_test_ds, get_class_weights,
    CLASS_NAMES, CLASS_TO_IDX, IDX_TO_CLASS,
    IMG_SIZE, BATCH_SIZE, SEED,
)

# reproducibility
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Num classes:", len(CLASS_NAMES))
print("Image size:", IMG_SIZE)

TensorFlow: 2.21.0
Num classes: 15
Image size: (224, 224)


In [2]:
CUSTOM_DIR   = PROJECT_ROOT / "custom_cnn"
LOGS_DIR     = CUSTOM_DIR / "logs"
PLOTS_DIR    = CUSTOM_DIR / "plots"
METRICS_DIR  = CUSTOM_DIR / "metrics"
CKPT_DIR     = CUSTOM_DIR / "best_model"

for d in [LOGS_DIR, PLOTS_DIR, METRICS_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

LOG_CSV      = LOGS_DIR / "training_log.csv"
CKPT_PATH    = CKPT_DIR / "best.keras"
METRICS_JSON = METRICS_DIR / "final_metrics.json"
CM_CSV       = METRICS_DIR / "confusion_matrix.csv"

print("Artifacts folder:", CUSTOM_DIR)

Artifacts folder: d:\Dev\College\GenAI\bangladeshi-fish-cnn-classification\custom_cnn


In [3]:
train_ds = get_train_ds()
val_ds   = get_val_ds()
test_ds  = get_test_ds()

print("Train batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Val   batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test  batches:", tf.data.experimental.cardinality(test_ds).numpy())

Train batches: 122
Val   batches: 27
Test  batches: 27


## Build the CNN

**Vivek owns this cell.** A baseline architecture is provided so the notebook
is runnable. Replace or extend it if you want to experiment. Keep:

- input shape `(224, 224, 3)`,
- output layer `Dense(len(CLASS_NAMES), activation="softmax")`,
- a `Flatten` or `GlobalAveragePooling2D` before the classifier head.

Do **not** add pretrained weights here. This is the from-scratch model.